# MedRelay on IBM LinuxONE - Hybrid GenAI + ML + Temporal Evidence Graphs for Clinical Follow-up Obligations

**Notebook mode:** controlled by the environment variable `MEDRELAY_MODE` (`fast` = development subset, `full` = every available record). Run all cells top to bottom from a clean kernel.

## 1. Project introduction and research questions

**Problem.** An instruction being *issued* is not the same as it being *fulfilled*. "Repeat the blood test within 48 hours, have the physician review the result, and contact the patient if further action is needed" is a **chain** of dependent obligations. A recorded blood test says nothing about whether the result was reviewed, or the patient told.

**Architecture (implemented in `src/`, orchestrated here):**

```
clinical text / events -> GenAI extraction (local Granite via llama.cpp) -> schema + evidence-grounding validation -> structured obligations
  -> supervised action classifier (cross-check) -> Temporal Evidence Graph (NetworkX, deterministic state rules)
  -> streaming event matching (SQLite, idempotent) -> calibrated ML risk (unresolved at deadline) -> constraint-based alerts + explanations -> audit log
```

**Research question.** Can dependency-aware temporal evidence graphs plus hybrid AI predict unresolved clinical obligations *earlier and more accurately* than conventional independent-task monitoring? *(A proposed contribution that needs experimental validation - not a claim of being first.)*

**What is real and what is not (read this first).**

| item | status |
|---|---|
| Hardware measurements, latencies, throughput | measured on the machine that executes this notebook (section 2 states its architecture) |
| Follow-up outcomes (completed / overdue / unverified / censored) | **simulated** by a seeded generator from documented source events; results are *performance on simulated follow-up processes only* |
| Action-classification labels | **weak (rule-proposed, "silver")**; **no human-reviewed (gold) labels exist** |
| Obligation-extraction evaluation sets | a **developer-annotated** challenge set (annotator = implementer, not independent) and **synthetic-by-construction** template documents |
| Appointment no-show data | real de-identified appointment records; label = attendance, *not* follow-up completion |
| Everything | synthetic or de-identified; **not clinically validated; recommendations require qualified human review** |

In [ ]:
import os, sys, json, time, warnings, platform
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 100); pd.set_option("display.precision", 3)
from src.data_loader import load_config
cfg = load_config(); MODE = cfg["mode"]
T_NB0 = time.time()
import psutil
_RSS_LOG = ROOT / "logs" / "rss_by_cell.csv"; _RSS_LOG.parent.mkdir(exist_ok=True); _RSS_LOG.write_text("cell,rss_mb,t_s\n")
def _log_rss(result):                                       # memory of this kernel after every cell (the 4 GB VM has only ~2.7 GB usable)
    with open(_RSS_LOG, "a") as fh:
        fh.write(f"{result.execution_count},{psutil.Process().memory_info().rss / 1e6:.0f},{time.time() - T_NB0:.0f}\n")
get_ipython().events.register("post_run_cell", _log_rss)
print(f"MedRelay | mode={MODE} | seed={cfg['seed']} | python {platform.python_version()} | machine {platform.machine()} | cwd {ROOT}")

## 2. IBM LinuxONE hardware verification

In [ ]:
from src import benchmarks as bm
hw = bm.hardware_report()
slim = {k: v for k, v in hw.items() if k not in ("lscpu", "free_h", "df_h")}
Path("results").mkdir(exist_ok=True)
Path("results/hardware.json").write_text(json.dumps(slim, indent=2, default=str))
if hw["is_s390x"]:
    display(HTML("<div style='padding:8px;background:#e8f5e9;border-left:5px solid #2e7d32'><b>CONFIRMED:</b> this notebook is executing on IBM Z / LinuxONE "
                 f"(<code>uname -m</code> = <b>s390x</b>, {hw['byteorder']}-endian, hypervisor <b>{hw['hypervisor']}</b>).</div>"))
else:
    display(HTML(f"<div style='padding:8px;background:#ffebee;border-left:5px solid #c62828'><b>WARNING:</b> architecture is {hw['uname_m']}, not s390x. "
                 "Every measurement below comes from THIS machine and is NOT a LinuxONE result.</div>"))
print(hw["uname_a"]); print(); print(hw["lscpu"][:2200]); print("\n/proc/cpuinfo features:", hw["cpu_features"])
keys = ["uname_m", "byteorder", "nproc", "memory_total_gb", "memory_available_gb", "disk_total_gb", "disk_free_gb", "python", "hypervisor", "has_vxe2", "nnpa_cpu_flag", "nnpa_facility_165_listed", "docker", "podman"]
display(pd.DataFrame({"value": [slim[k] for k in keys]}, index=keys))
print(hw["free_h"]); print(hw["df_h"])

In [ ]:
nn = bm.nnpa_probe()
display(pd.DataFrame({"value": nn}).astype(str))
print("NNPA is PRESENT in the CPU flags/facility list" if nn["cpu_flag_nnpa"] else "NNPA flag not present on this host")
print("Presence is not use: tree/linear models (sklearn, XGBoost) do NOT run on NNPA. A device-level zDNN test is run in section 16.")

## 3. Environment and dependency configuration

In [ ]:
from src.llm_backend import gguf_byteorder, host_byteorder
display(pd.DataFrame([{"package": k, "version": v} for k, v in hw["packages"].items()]))
mods = sorted((ROOT / cfg["llm"]["models_dir"]).glob("*.gguf"))
display(pd.DataFrame([{"model_file": m.name, "size_mb": round(m.stat().st_size / 1e6, 1), "gguf_byteorder": gguf_byteorder(m), "host_byteorder": host_byteorder(),
                       "usable_without_conversion": gguf_byteorder(m) == host_byteorder()} for m in mods]))
import llama_cpp
print("llama.cpp via llama-cpp-python", llama_cpp.__version__); print(llama_cpp.llama_print_system_info().decode())
print("\nSelected configuration (config.yaml):"); print(open("config.yaml").read())
print("Remote LLM backends:", "ENABLED" if cfg["llm"]["allow_remote"] else "disabled (all GenAI inference is local; no clinical text leaves this machine)")

**Installation approach (see `docs/linuxone_setup.md`):** no Docker/Podman is required. Scientific Python comes from the distribution's precompiled `s390x` packages (PyPI has no `s390x`
wheel for NumPy/pandas/scikit-learn/pyarrow), `llama-cpp-python` is compiled from source for the z16 instruction set (VXE), Hugging Face GGUF files (little-endian) are converted once to
big-endian with `gguf-convert-endian`. Because `pyarrow` is unavailable, the dataset exports are consumed as gzip CSV.

## 4. Dataset loading and integrity validation

In [ ]:
from src.data_loader import validate_datasets, build_stream, load_stream_truth
t = time.time()
rep = validate_datasets(cfg)
display(pd.DataFrame([{"check": k, "passed": v["ok"], "detail": str(v["detail"])[:120]} for k, v in rep["checks"].items()]))
print("hard failures:", rep["hard_failures"] or "none"); print("warnings:", *rep["warnings"], sep="\n  - ")
assert not rep["hard_failures"], rep["hard_failures"]
print("\nrecord counts:", rep["counts"]); print("validation took", round(time.time() - t, 1), "s")

In [ ]:
prov = rep["label_provenance"]
display(pd.DataFrame([{"dataset": "action classification", "label source": "weak rule-proposed (silver) + 622 train-only template (synthetic)", "human-reviewed gold": prov["gold_human_reviewed_action_labels"]},
                      {"dataset": "follow-up risk", "label source": prov["risk_labels"], "human-reviewed gold": 0},
                      {"dataset": "appointment no-show", "label source": prov["noshow_labels"], "human-reviewed gold": "n/a (real attendance record)"},
                      {"dataset": "evidence graph / event stream", "label source": "simulated scenario ground truth (hidden file, never in live payloads)", "human-reviewed gold": 0}]))
print("action label tiers:", prov["action_label_tiers"]); print("action label methods:", prov["action_label_methods"]); print("action sources:", prov["action_sources"])
cb = rep["class_balance"]
display(pd.DataFrame(cb["action"]).T.rename_axis("split")); display(pd.DataFrame(cb["risk_binary"]).T.rename_axis("split").rename(columns={"1": "overdue(1)", "0": "completed(0)"}))
display(pd.DataFrame(cb["risk_outcomes"]).T.rename_axis("split")); display(pd.DataFrame(cb["noshow"]).T.rename_axis("split"))

In [ ]:
from src.risk_predictor import build_graph_from_stream, feature_parity
from src.data_loader import subset_patients, read_source
pats = read_source(cfg, "interim/patients.csv.gz")
keep = subset_patients(cfg, pats)
t = time.time(); G0 = build_graph_from_stream(cfg, keep)
par = feature_parity(cfg, G0, sample=4000, patients=keep)
print("replay-stream graph:", G0.stats()); print("feature parity (graph recomputation vs dataset-pipeline tables):", par, "| %.1f s" % (time.time() - t))
assert par["exact_or_within_tolerance"], par
stream_paths = build_stream(cfg)
from collections import Counter
from src.data_loader import iter_stream
ev_counts, n_deliv, n_dup, n_late = Counter(), 0, 0, 0
from src.evidence_graph import to_epoch as _te
for ev_ in iter_stream(cfg):
    ev_counts[ev_["event_type"]] += 1; n_deliv += 1; n_dup += ev_["delivery_attempt"] > 1; n_late += (_te(ev_["ingest_time_utc"]) - _te(ev_["event_time_utc"])) > 7200
display(pd.DataFrame([ev_counts]).T.rename(columns={0: "deliveries"}).T)
an = pd.read_csv(stream_paths["anomalies"])
print("delivery imperfections injected into the replay stream:", an.anomalies.value_counts().to_dict(), "| deliveries:", n_deliv, "| duplicates:", n_dup, "| delayed >2h:", n_late)
del G0
import gc; gc.collect()

## 5. Exploratory data analysis

In [ ]:
from src.data_loader import load_action, load_risk_hidden, load_noshow, load_interim
from src import visualization as viz
T = load_interim(cfg); A = load_action(cfg); NS = load_noshow(cfg); GT = load_stream_truth(cfg)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ct = pd.crosstab(T["obligations"].category, T["ground_truth"].set_index("obligation_id").loc[T["obligations"].obligation_id].outcome_label.values)
ct.plot(kind="barh", stacked=True, ax=ax[0], width=.8); ax[0].set_title("Obligations by category and simulated outcome"); ax[0].legend(fontsize=7)
act = pd.DataFrame({sp: A[sp][[c for c in A[sp].columns if c.startswith("y_")]].sum() for sp in ("train", "validation", "test")}); act.index = [i[2:] for i in act.index]
act.plot(kind="barh", ax=ax[1]); ax[1].set_title("Action labels per split (silver)"); ax[1].set_xscale("log")
ns = NS.assign(band=pd.cut(NS.wait_days, [-1, 0, 3, 7, 14, 30, 400])).groupby("band").no_show.agg(["mean", "size"])
ax[2].bar(ns.index.astype(str), ns["mean"]); ax[2].set_title("No-show rate by waiting time (real data)"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); viz.save(fig, cfg, "eda_overview"); plt.show()

In [ ]:
ob = T["obligations"]
win = (pd.to_datetime(ob.due_time_utc) - pd.to_datetime(ob.created_time_utc)).dt.total_seconds() / 86400
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].hist(win.clip(upper=60), bins=40); ax[0].set_title("Deadline window (days, clipped at 60)")
depth = ob.groupby("chain_id").size().value_counts().sort_index(); ax[1].bar(depth.index.astype(str), depth.values); ax[1].set_title("Dependency chain length (obligations per chain)")
src = pd.concat([T["patients"].groupby(["source_dataset", "split"]).size().unstack(fill_value=0)]); src.plot(kind="bar", stacked=True, ax=ax[2]); ax[2].set_title("Patients per source and split"); ax[2].tick_params(axis="x", rotation=25)
plt.tight_layout(); viz.save(fig, cfg, "eda_structure"); plt.show()
print("encounters:", len(T["encounters"]), "| patients:", len(T["patients"]), "| obligations:", len(ob), "| chains:", ob.chain_id.nunique(), "| source events:", len(T["events"]))
print("share of obligations whose first step is a documented source event vs rule-extracted vs simulated:")
display(ob.groupby("seed_kind").size().rename("obligations").to_frame().T)
display(HTML("<i>Time precision: Synthetic Hospital dates are month-granular (all on the 15th); sub-day times are imputed. Synthea has clock times.</i>"))

## 6. Clinical obligation schema

In [ ]:
import json as _json
from src.clinical_extractor import ExtractedObligation, EvidenceSpan
from src.ontology import STEP_SPEC, COMPLETION_EVENT, State, NODE_TYPES, EDGE_TYPES
sch = ExtractedObligation.model_json_schema()
print(_json.dumps({k: v for k, v in sch["properties"].items()}, indent=1)[:2600])
display(pd.DataFrame([{"workflow step": s, "category": c, "role": r, "completed only by event": COMPLETION_EVENT[s]} for s, (c, r) in STEP_SPEC.items()]))
display(pd.DataFrame([{"state": s.value} for s in State]).T)
print("graph node types:", NODE_TYPES); print("graph edge types:", EDGE_TYPES)

In [ ]:
from pydantic import ValidationError
bad_examples = [dict(task_type="laboratory_test", due_interval="two_days"), dict(task_type="telepathy"), dict(task_type="imaging", responsible_role="wizard")]
for kw in bad_examples:
    try:
        ExtractedObligation(obligation_id="x", description="x test", evidence_span=EvidenceSpan(start=0, end=1, text="x"), source_note_id="n", confidence=.5, **kw)
    except ValidationError as e:
        print("REJECTED:", kw, "->", e.errors()[0]["msg"][:110])

## 7. GenAI obligation extraction

A local **IBM Granite 4.0** instruction model (GGUF, run with llama.cpp on this machine; Apache-2.0) converts text into schema-validated obligations. Four extraction strategies are compared:

| id | strategy |
|---|---|
| **A** | deterministic rule-based clause decomposition (no ML) |
| **B** | small local GenAI, grammar-constrained JSON, accepted after schema parsing only |
| **C / C2** | B + validation: evidence span must be verbatim in the source, unsupported deadlines/roles/conditionals are removed, dependencies must be acyclic; **C2** also fills missing deadline/role/dependency fields *only through deterministic parsers over the cited span* |
| **E** | rules + validated GenAI, overlapping obligations merged (rule fields win) |

**Evaluation data (no human-reviewed gold exists):** (i) 40 **developer-annotated** instruction texts (annotator = implementer; not independent), (ii) **template documents with exact ground truth by construction** whose sentence structures are *held out* from everything the rules and prompts were written against.

In [ ]:
from src.experiments import Lab
from src.exp_extraction import run_exp1, cached_backend, make_extractors
from src.scenario_lab import DEMO_TEXT
from src.clinical_extractor import RuleExtractor
lab = Lab(cfg)
print("fast/full parameters:", lab.p)
SPEC_TEXT = ("Patient should undergo repeat blood testing in two days. Results must be reviewed by the treating physician and follow-up arranged if abnormalities persist.")
be = cached_backend(lab, "primary")
print(f"GenAI model: {be.name} | size {be.size_bytes/1e6:.0f} MB | threads {be.n_threads} | load {be.load_s:.2f} s")
exts = make_extractors(be)
def show(res):
    rows = [{"id": o.obligation_id.split(":")[-1], "task_type": o.task_type, "due": o.due_interval, "role": o.responsible_role, "depends_on": [d.split(":")[-1] for d in o.dependencies],
             "conditional": o.conditional_requirements, "evidence_span": o.evidence_span.text[:70], "grounded": o.evidence_span.grounded, "missing": o.missing_information} for o in res.obligations]
    return pd.DataFrame(rows)
for name in ("A_rules", "B_llm_raw", "C2_llm_validated_filled", "E_rules_plus_llm"):
    r = exts[name].extract(SPEC_TEXT, "spec")
    print(f"--- {name}: {len(r.obligations)} obligations | latency {r.latency_s:.1f}s | schema_valid={r.schema_valid} | rejected={[x['reason'] for x in r.rejected]} | repairs={[(x.get('field'), x['action']) for x in r.repairs]}")
    display(show(r))

In [ ]:
t = time.time()
E1 = run_exp1(lab, "primary", include=["A_rules", "B_llm_raw", "C_llm_validated", "C2_llm_validated_filled", "E_rules_plus_llm"])
cols = ["extractor", "corpus", "n_docs", "typed_precision", "typed_recall", "typed_f1", "typed_f1_lo", "typed_f1_hi", "det_f1", "deadline_acc", "spurious_deadlines", "role_acc", "cond_acc",
        "dep_f1_on_matched", "dep_matched_pairs", "unsupported_field_rate", "oblig_with_unsupported", "schema_valid", "latency_mean_s", "latency_p95_s"]
display(E1["table"][cols]); print("Experiment 1 took", round(time.time() - t), "s | LLM cache hits/misses:", E1["backend"].hits, E1["backend"].misses)
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
for k, (col, ttl) in enumerate((("typed_f1", "Obligation F1 (task type + span)"), ("unsupported_field_rate", "Unsupported-fact rate (lower is better)"), ("latency_mean_s", "Latency per document (s)"))):
    pv = E1["table"].pivot(index="extractor", columns="corpus", values=col); pv.plot(kind="bar", ax=ax[k]); ax[k].set_title(ttl); ax[k].tick_params(axis="x", rotation=30); ax[k].legend(fontsize=7)
plt.tight_layout(); viz.save(fig, cfg, "exp1_extraction"); plt.show()

In [ ]:
cat = E1["per_category"]
display(cat[cat.corpus == "developer_annotated_challenge"].pivot(index="category", columns="extractor", values="f1").round(2).assign(support=cat[(cat.corpus == "developer_annotated_challenge") & (cat.extractor == "A_rules")].set_index("category").eval("tp + fn")))
display(cat[cat.corpus == "template_heldout_phrasing"].pivot(index="category", columns="extractor", values="f1").round(2).assign(support=cat[(cat.corpus == "template_heldout_phrasing") & (cat.extractor == "A_rules")].set_index("category").eval("tp + fn")))
# what validation actually removed from raw GenAI output
rej, rep_ = {}, {}
for (name, corpus), res in E1["results"].items():
    if name == "C_llm_validated":
        for r in res:
            for x in r.rejected: rej[x["reason"]] = rej.get(x["reason"], 0) + 1
            for x in r.repairs: rep_[f"{x.get('field')}:{x['action']}"] = rep_.get(f"{x.get('field')}:{x['action']}", 0) + 1
print("validation rejected (obligations):", rej); print("validation repaired (fields):", rep_)

In [ ]:
big_ok = (ROOT / cfg["llm"]["models_dir"] / cfg["llm"]["large"].replace(".gguf", ".be.gguf")).exists() or (ROOT / cfg["llm"]["models_dir"] / "original_le" / cfg["llm"]["large"]).exists()
if big_ok:
    nl = lab.p["llm_large_docs"]
    t = time.time(); E1L = run_exp1(lab, "large", n_chal=nl["challenge"], n_tpl=nl["template"], include=["A_rules", "B_llm_raw", "C2_llm_validated_filled", "E_rules_plus_llm"])
    display(E1L["table"][["extractor", "model", "corpus", "n_docs", "typed_precision", "typed_recall", "typed_f1", "det_f1", "deadline_acc", "unsupported_field_rate", "schema_valid", "latency_mean_s", "latency_p95_s"]])
    print("larger-model comparison took", round(time.time() - t), "s on a SMALLER sample (n_docs above) because generation is ~5x slower")
    from src.llm_backend import release_backend
    release_backend(cfg, "large")                      # the 1B model held ~1.8 GB of the VM's ~2.7 GB usable RAM for the rest of the notebook
    print("larger model released from memory; kernel RSS now %.0f MB" % (psutil.Process().memory_info().rss / 1e6))
else:
    print("larger local model not installed; skipping the upper-bound comparison")

In [ ]:
# Model selection from MEASURED quality (not assumed): average typed-F1 over both corpora, restricted to GenAI-based extractors, plus latency.
sel = E1["table"][E1["table"].extractor.isin(["B_llm_raw", "C2_llm_validated_filled", "E_rules_plus_llm"])].groupby("extractor")[["typed_f1", "unsupported_field_rate", "latency_mean_s"]].mean()
display(sel)
best_ext = sel.typed_f1.idxmax(); print("highest mean typed F1 among GenAI-based extractors (350M):", best_ext)
rules_f1 = E1["table"][E1["table"].extractor == "A_rules"].typed_f1.mean()
print(f"rule-based baseline mean typed F1 = {rules_f1:.3f}; best GenAI-based = {sel.typed_f1.max():.3f}")
print("DECISION: Granite-4.0-350M (Q8_0) alone is too weak to replace the rules (it under-extracts, omits deadlines and fabricates roles). It is kept only behind the validator and in the rules+GenAI ensemble (E), which adds the "
      "obligations the rules miss while every field stays evidence-grounded. The 1B model (if present above) is the measured upper-bound comparison.")
SELECTED_EXTRACTOR = "E_rules_plus_llm"

## 8. ML clinical action classification (Model A)

TF-IDF (word + character n-grams) with **logistic regression**, **linear SVM** and a **compact neural net** (TF-IDF -> SVD -> MLP). Hyper-parameters are chosen by random search on *train -> validation*
macro-F1 (budget in `config.yaml`); the test split is used once. Labels are **silver** (rule-proposed). `challenge_span_*` columns score the same classifier on **developer-annotated** obligation spans,
a different label source, to show how much the silver agreement overstates correctness.

In [ ]:
from src.experiments import model_a_action_classifier
t = time.time(); A_res = model_a_action_classifier(lab); print("Model A took", round(time.time() - t), "s")
display(A_res["comparison"].round(3)); print("best family:", A_res["best_kind"], A_res["best_params"])
display(A_res["per_class_test"].round(3)); print(A_res["synthetic_augmentation"]); print(A_res["label_provenance"])
from IPython.display import Image
for nm in ("model_a_confusion_silver", "model_a_confusion_spans"):
    display(Image(filename=str(ROOT / "figures" / f"{nm}.png")))

In [ ]:
from src.evaluation import type_accuracy_on_detected
rows = []
chal = E1["corpora"]["developer_annotated_challenge"]
for (name, corpus), res in E1["results"].items():
    if corpus == "developer_annotated_challenge":
        rows.append({"system": f"extraction {name}", **type_accuracy_on_detected(chal, res)})
rows.append({"system": f"ML classifier ({A_res['best_kind']}) on the TRUE spans", "n_detected": int(sum(len(d['obligations']) for d in chal)),
             "type_accuracy": float(A_res["comparison"].set_index("model").loc[A_res["best_kind"], "challenge_span_accuracy"])})
display(pd.DataFrame(rows).round(3))
print("Fair-comparison note: both rows score the CATEGORY assigned to a span against developer annotations. The classifier is given the gold span; the extractors must also find the span "
      "(restricted here to spans they detected), so this favours neither strongly. Silver-test agreement is NOT correctness.")

## 9. Temporal evidence graph construction

Node types (Patient, Encounter, ClinicalNote, Obligation, Task, Result, Review, Acknowledgment, Notification, EvidenceEvent, ResponsibleRole), edge types (EXTRACTED_FROM, DEPENDS_ON, ASSIGNED_TO,
EVIDENCED_BY, COMPLETED_BY, REQUIRES_REVIEW, CONTRADICTS, BELONGS_TO). Workflow states come from deterministic rules R1-R7 that cite their evidence. The cells below *demonstrate* each required semantic on the scripted episode.

In [ ]:
from src.event_processor import EventProcessor
from src.evidence_graph import TemporalEvidenceGraph, iso, to_epoch
from src.scenario_lab import demo_events, demo_obligations, T0
from src import visualization_demo as vd
H = 3600.0
PID = "demo:patient-001"
gp = EventProcessor(cfg, ":memory:", scorers={}, build_nx=True)
gp.run(demo_events("delayed_ack")); gp.finalize()
g = gp.graph
print("graph:", g.stats())
from collections import Counter
print("node types:", dict(Counter(d["ntype"] for _, d in g.G.nodes(data=True)))); print("edge types:", dict(Counter(d["etype"] for _, _, d in g.G.edges(data=True))))
for when, hours in (("after the lab test (+31h)", 31), ("after deadline of the review, tracker open (+122h)", 122), ("after the delayed acknowledgment is documented (+152h)", 152)):
    now = T0 + hours * H
    print("\n", when); display(pd.DataFrame([{"obligation": o.split(":")[-1], "step": g.obs[o].step_type, **{k: v for k, v in dict(state=g.derive_state(o, now, live=True).state.value, rule=g.derive_state(o, now, live=True).rule,
                                         evidence=g.derive_state(o, now, live=True).evidence_ids).items()}} for o in sorted(g.obs)]))

In [ ]:
fig = vd.plot_evidence_graph(gp, PID, now=T0 + 31 * H, title="Graph at +31h: the TEST is completed with evidence, the chain is NOT"); viz.save(fig, cfg, "graph_t31"); plt.show()
fig = vd.plot_evidence_graph(gp, PID, now=T0 + 152 * H, title="Graph at +152h: delayed acknowledgment (dotted red) reconciled the review obligation"); viz.save(fig, cfg, "graph_t152"); plt.show()

In [ ]:
# semantic checks, each executed live
from tests.conftest import mk_ev, mk_ob   # test helpers (synthetic records)
t0 = to_epoch("2024-03-04T09:00:00Z")
g1 = TemporalEvidenceGraph(12)
g1.ingest(mk_ob("o0", "lab_collected", t0, t0 + 48 * H, [], pos=0, n=3)); g1.ingest(mk_ob("o1", "result_reviewed", t0, t0 + 120 * H, ["o0"], pos=1, n=3)); g1.ingest(mk_ob("o2", "patient_notified", t0, t0 + 192 * H, ["o1"], pos=2, n=3))
g1.ingest(mk_ev("e1", "o0", "lab_collected", t0 + 10 * H))
checks = []
checks.append(("completed test does NOT complete the chain", [g1.derive_state(o, t0 + 20 * H).state.value for o in ("o0", "o1", "o2")]))
g1.ingest(mk_ev("e2", "o0", "result_posted", t0 + 14 * H)); g1.ingest(mk_ev("e3", "o1", "result_opened", t0 + 20 * H))
checks.append(("result available + opened is NOT an acknowledgment", (g1.derive_state("o1", t0 + 30 * H).state.value, g1.features("o1", t0 + 30 * H)["ack_state"])))
g1.ingest(mk_ev("e4", "o1", "result_acknowledged", t0 + 40 * H)); checks.append(("reviewed result does NOT notify the patient", (g1.derive_state("o1", t0 + 50 * H).state.value, g1.derive_state("o2", t0 + 50 * H).state.value)))
r = g1.ingest(mk_ev("e4", "o1", "result_acknowledged", t0 + 40 * H, t_ingest=t0 + 60 * H)); checks.append(("duplicate delivery ignored", (r.duplicate, g1.n_duplicates)))
g2 = TemporalEvidenceGraph(12); g2.ingest(mk_ob("a", "lab_collected", t0, t0 + H, ["b"])); rr = g2.ingest(mk_ob("b", "result_reviewed", t0, t0 + 2 * H, ["a"]))
checks.append(("cyclic dependency rejected", (rr.rejected, g2.rejected_cycles)))
g3 = TemporalEvidenceGraph(12); g3.ingest(mk_ob("x", "lab_collected", t0, t0 + 48 * H, [])); d = t0 + 48 * H
checks.append(("no evidence: grace -> unverified (not 'overdue')", [g3.derive_state("x", d + h * H).state.value for h in (1, 13)]))
g3.ingest(mk_ev("s", "x", "task_status_snapshot", d + H, details={"status": "open"})); checks.append(("tracker confirms open -> overdue", g3.derive_state("x", d + 2 * H).state.value))
g3.ingest(mk_ev("late", "x", "lab_collected", t0 + 40 * H, t_ingest=t0 + 90 * H)); checks.append(("late-documented completion reconciles (live view)", (g3.derive_state("x", d + 10 * H, live=True).state.value, g3.derive_state("x", d + 50 * H, live=True).state.value)))
g3.ingest(mk_ev("c", "x", "conflicting_record", t0 + 45 * H, ver="unverified")); checks.append(("contradictory evidence", g3.derive_state("x", d + 50 * H, live=True).state.value))
display(pd.DataFrame(checks, columns=["property", "observed"]))

## 10. Follow-up risk prediction (Model B) - Experiment 2

Target: probability that an obligation **remains unresolved at its deadline**, from features available at the assessment time only (feature code = the evidence graph; events after the cutoff are invisible).
Training uses **dense snapshots** at 8 points across each obligation's window (the dataset pipeline's single 20-80 % assessment point is also reported as `orig_protocol_*`), only for obligations still open at the snapshot.
**Unverified and censored obligations are never treated as failures.** Families: logistic regression, HistGradientBoosting, random forest, XGBoost; feature sets: *task-only*, *B independent tasks* (no dependency/evidence-graph features), *C graph-enhanced*.
All labels are **simulated** follow-up outcomes.

In [ ]:
from src.experiments import model_b_followup_risk
t = time.time(); B_res = model_b_followup_risk(lab); print("Model B took", round(time.time() - t), "s")
cols = ["model", "features", "val_pr_auc", "test_pr_auc", "test_pr_auc_ci_lo", "test_pr_auc_ci_hi", "test_roc_auc", "test_brier", "test_ece", "test_brier_uncalibrated", "test_ece_uncalibrated", "orig_protocol_test_pr_auc"]
display(B_res["comparison"][cols].round(3)); display(B_res["baselines"].round(3))
print("best per feature set:", B_res["summary"]["best_per_featureset"]); print("paired patient-bootstrap differences in PR-AUC:", B_res["summary"]["paired_bootstrap"])
display(B_res["thresholds"].round(3)); display(Image(filename=str(ROOT / "figures" / "model_b_calibration.png")))

In [ ]:
from src.risk_predictor import group_permutation_importance, binary
from src.ontology import FEATURE_GROUPS
te, yte = binary(lab.dense()["test"])
mC = B_res["best"]["C_graph"][0]
gi = group_permutation_importance(mC, te, yte, FEATURE_GROUPS, repeats=5 if MODE == "fast" else 10, seed=lab.seed)
display(gi.round(4))
ax = gi.set_index("feature_group").pr_auc_drop_mean.plot(kind="barh", xerr=gi.set_index("feature_group").pr_auc_drop_std, figsize=(6, 3)); ax.set_title(f"Group permutation importance ({mC.name}) - PR-AUC drop"); plt.tight_layout(); viz.save(plt.gcf(), cfg, "model_b_group_importance"); plt.show()

### Secondary model C: appointment no-show on **real** data

The only real (non-simulated) outcome in the project is appointment attendance (public Kaggle no-show data, de-identified). It is a proof of concept that the same calibrated-model recipe works on real scheduling data;
the label is **attendance, not follow-up completion**, and the model must never be used to deny or deprioritise care. A scheduling-only feature set is compared with one that adds demographics (which can encode bias).

In [ ]:
from src.experiments import model_c_noshow
t = time.time(); C_res = model_c_noshow(lab); print("Model C took", round(time.time() - t), "s | n:", C_res["n"], "| test prevalence %.3f" % C_res["prevalence_test"])
display(C_res["comparison"].round(3)); print("selected on validation PR-AUC:", C_res["selected"]); display(C_res["subgroups"].round(3)); print(C_res["caveats"])
display(Image(filename=str(ROOT / "figures" / "model_c_calibration.png")))

## 11. Hybrid GenAI + ML integration

The integration layer (`src/pipeline.py`) wires the pieces: the extractor *proposes* validated obligation records; the supervised classifier cross-checks the task type (strategy **D**); the evidence graph
turns records plus streaming events into deterministic states; the calibrated model forecasts risk. **The LLM never sets a workflow state.**

In [ ]:
from src.exp_replay import load_scorers, replay, split_patients
t = time.time(); scorers = load_scorers(lab, ablations=True); print("scorers:", list(scorers), "| trained/loaded in", round(time.time() - t), "s")
E1D = run_exp1(lab, "primary", include=["A_rules", "C2_llm_validated_filled", "D_hybrid_llm_ml", "E_rules_plus_llm"])
display(E1D["table"][["extractor", "corpus", "typed_precision", "typed_recall", "typed_f1", "det_f1", "deadline_acc", "unsupported_field_rate", "latency_mean_s"]].round(3))
ov = sum(1 for (n, c), res in E1D["results"].items() if n == "D_hybrid_llm_ml" for r in res for x in r.repairs if x.get("action") == "overridden_by_classifier")
print("task_type overrides made by the ML classifier in strategy D:", ov)

In [ ]:
from src.pipeline import HybridPipeline, script_events_for_created
pipe = HybridPipeline(cfg, make_extractors(cached_backend(lab, "primary"), E1D["classifier"])[SELECTED_EXTRACTOR], {"B_independent_ml": scorers["B_independent_ml"], "C_graph_ml": scorers["C_graph_ml"]}, build_nx=True)
res, created = pipe.ingest_note(DEMO_TEXT, "demo-note-1", "demo:patient-001")
print("extractor:", res.extractor, "| obligations:", len(res.obligations), "| latency %.1fs" % res.latency_s); print("DEMO TEXT:", DEMO_TEXT)
display(pd.DataFrame([{"obligation": o.obligation_id.split(":")[-1], "task_type": o.task_type, "due": o.due_interval, "role": o.responsible_role, "depends_on": [d.split(":")[-1] for d in o.dependencies],
                       "conditional": o.conditional_requirements, "span": o.evidence_span.text, "missing_information": o.missing_information, "confidence": round(o.confidence, 2)} for o in res.obligations]))
print("monitored obligations after extraction (all pending until evidence arrives):"); display(pipe.report()[["obligation", "step", "deadline", "depends_on", "state", "risk"]])

In [ ]:
script = script_events_for_created(created, "delayed_ack")
for k in (3, 6, len(script)):
    pipe2 = HybridPipeline(cfg, make_extractors(cached_backend(lab, "primary"), E1D["classifier"])[SELECTED_EXTRACTOR], {"B_independent_ml": scorers["B_independent_ml"], "C_graph_ml": scorers["C_graph_ml"]})
    pipe2.ingest_note(DEMO_TEXT, "demo-note-1", "demo:patient-001"); pipe2.feed(script[:k]);
    pipe2.proc._advance(to_epoch(script[k - 1]["ingest_time_utc"]) + 1); pipe2.proc.flush()
    print(f"\nafter {k}/{len(script)} streamed events:"); display(pipe2.report()[["step", "state", "late", "risk", "rule"]])
print("alerts raised:", [(a.rule, a.status) for a in pipe2.proc.alert_log][:8])

In [ ]:
from src.alert_engine import verbalize_alert
a = next((x for x in pipe2.proc.alert_log), None)
if a is not None:
    print("DETERMINISTIC, evidence-grounded explanation (record of truth):\n", a.explanation)
    v = verbalize_alert(cached_backend(lab, "primary").backend, a)
    print("\nOptional GenAI paraphrase | grounded =", v["grounded"], "| foreign tokens:", v["foreign_tokens"], "| %.1fs" % v["latency_s"]); print(v["llm_text"]); print("\nDISPLAYED:", v["text"][:600])

## 12. Real-time event processing

The replay engine streams the processed event stream (held-out **test** patients) through `EventProcessor`: per delivery it (1) validates schema/provenance, (2) resolves patient/obligation, (3) updates the temporal evidence graph,
(4) recomputes affected states, (5) builds current-time features, (6) scores risk with the live models, (7) applies deterministic alert rules, (8) builds an evidence-grounded explanation, (9) appends to the audit log.
State is persisted in **SQLite (WAL)**; processing is **idempotent**; event time and ingestion time are kept separately; the stream contains late, duplicated and out-of-order deliveries.

In [ ]:
from src.exp_replay import replay, split_patients, truth_frame
from src.event_processor import EventProcessor, deadline_only_score
from src.data_loader import iter_stream
db_path = ROOT / cfg["paths"]["db"]
for suf in ("", "-wal", "-shm"):
    (Path(str(db_path) + suf)).unlink(missing_ok=True)
pt = split_patients(lab, "test")
events_test = list(iter_stream(cfg, patients=pt))
t = time.time()
proc = replay(lab, pt, scorers, events=events_test, db=str(db_path), rescore_batch=1, track_latency=True, build_nx=True)
wall = time.time() - t
lat = np.array(proc.latencies) * 1000
print(f"replayed {len(events_test)} deliveries for {len(pt)} test patients in {wall:.1f}s -> {len(events_test)/wall:.0f} deliveries/s with per-event streaming inference (3 risk scorers) and SQLite persistence")
print("per-delivery latency ms p50/p95/p99:", np.percentile(lat, [50, 95, 99]).round(2)); print("processor counters:", dict(proc.stats))
print("graph:", proc.graph.stats()); print("final workflow states:", proc.state_counts()); print("database integrity:", proc.integrity_check())

In [ ]:
al = proc.alerts_df(); display(al.rule.value_counts().rename("alerts").to_frame().T); display(al.head(5))
display(proc.audit_df().head(8)[["oid", "t_ingest", "old_state", "new_state", "rule", "trigger_event_id"]])
print("example grounded alert:\n", next(a for a in proc.alert_log if a.rule == "RESULT_NOT_ACKNOWLEDGED").explanation)

In [ ]:
# idempotency: re-delivering events changes nothing
before = {o: (s[0], s[2]) for o, s in proc.states.items()}; n_tr = len(proc.transitions)
proc.run(events_test[:3000]); proc.finalize()
after = {o: (s[0], s[2]) for o, s in proc.states.items()}
print("re-delivered 3000 events: states changed =", sum(before[o] != after[o] for o in before), "| new transitions =", len(proc.transitions) - n_tr, "| duplicates ignored so far =", proc.graph.n_duplicates)
# crash recovery: rebuild from the persisted event log only
proc.commit()
t = time.time(); rec = EventProcessor.recover(cfg, db_path, scorers={}); 
same = np.mean([rec.states[o][0] == proc.states[o][0] for o in proc.states if o in rec.states])
print(f"recovered {rec.graph.stats()['obligations']} obligations / {rec.graph.stats()['events_ingested']} events from SQLite in {time.time()-t:.1f}s; stored states restored identically: {same:.3f}")
# reproducibility: a second replay of the same stream gives identical audit trails
p2 = replay(lab, pt, scorers, events=events_test, rescore_batch=1, build_nx=False)
sig = lambda p: [(t["oid"], t["new_state"], t["rule"]) for t in p.transitions]
print("replay reproducibility (identical transition sequence in a second, independent run):", sig(proc) == sig(p2), f"| {len(sig(p2))} transitions")

## 13. Closed-loop evidence verification

In [ ]:
from src.exp_replay import run_exp3, run_exp4, verification_metrics
t = time.time(); E3 = run_exp3(lab, scorers); print("Experiment 3 replay+evaluation took", round(time.time() - t), "s"); print(E3["n"])
se = E3["state_eval"]
display(pd.DataFrame({k: {"accuracy at t*=deadline+grace+1h": v["accuracy"], **{f"recall[{c}]": r for c, r in v["per_class_recall"].items()}} for k, v in se.items()}).T.round(3))
display(se["engine_evidence_aware"]["confusion"]); display(se["naive_absence_is_failure"]["confusion"])
print("t* is evaluated using only evidence INGESTED by t*; 'engine_final_after_reconciliation' = same engine after all late evidence arrived.")

In [ ]:
VM = verification_metrics(lab, E3["processor_test"], truth_frame(lab, E3["processor_test"]), list(iter_stream(cfg, patients=split_patients(lab, "test"))))
display(pd.DataFrame({"value": VM}).astype(str))

In [ ]:
t = time.time(); E4 = run_exp4(lab, rate=0.3); print("Experiment 4 took", round(time.time() - t), "s")
display(E4["table"].round(3).T)
print("Reading: 'missing_completion' - the task was done but its evidence was deleted: the evidence-aware engine must NOT call it a confirmed failure (naive deadline logic does). 'delayed_evidence' - interim state wrongly "
      "overdue under the naive rule, reconciled by the engine. 'contradictory' - naive logic counts it completed. 'unacknowledged_results' - the engine alerts before the deadline. 'duplicates' - states unchanged.")

## 14. Baseline comparisons and ablations

**Experiment 3 (main).** Three systems on the *same* replayed stream of held-out patients: **A** deadline-only reminders, **B** independent tasks + ML, **C** dependency-aware evidence graph + ML.
Operating thresholds are chosen on *validation* patients for a target recall and applied unchanged to *test* patients. The key early-warning requirement is *minimum lead*: alerts must be raised at least 0/24/72 h before the deadline.

In [ ]:
ew = E3["early_warning"]; sy = E3["summary"]
display(sy.round(3))
for lead in (24.0, 72.0):
    print(f"--- alerts required at least {lead:.0f} h before the deadline; thresholds from VALIDATION at recall target 0.7, applied to TEST")
    display(ew[(ew.min_lead_h == lead) & (ew["target_recall(val)"] == 0.7)][["system", "threshold(val)", "alerts", "recall", "precision", "false_alert_rate", "median_lead_h", "alerts_on_unverified_or_censored"]].round(3))
print("paired patient-bootstrap differences in max-score PR-AUC:", E3["paired"])

In [ ]:
sub = ew[(ew["target_recall(val)"] == 0.7) & ew.system.isin(["A_deadline_only", "A_reminder_24h", "B_independent_ml", "C_graph_ml"])]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
sy2 = sy[sy.system.isin(["A_deadline_only", "A_reminder_24h", "B_independent_ml", "C_graph_ml"])].set_index("system")
sy2[["pr_auc_at_T-72h", "pr_auc_at_T-24h", "pr_auc_max_score"]].plot(kind="bar", ax=ax[0]); ax[0].set_title("PR-AUC of the score known at T-72h / T-24h / max before deadline"); ax[0].tick_params(axis="x", rotation=20); ax[0].legend(fontsize=7)
pv = sub.pivot(index="system", columns="min_lead_h", values="false_alert_rate"); pv.plot(kind="bar", ax=ax[1]); ax[1].set_title("False-alert rate at recall target 0.7 (val), by required lead (h)"); ax[1].tick_params(axis="x", rotation=20)
pv = sub.pivot(index="system", columns="min_lead_h", values="recall"); pv.plot(kind="bar", ax=ax[2]); ax[2].set_title("Achieved TEST recall at the validation-chosen threshold"); ax[2].tick_params(axis="x", rotation=20)
plt.tight_layout(); viz.save(fig, cfg, "exp3_systems"); plt.show()
display(E3["alert_table"].round(2)); print("Engine alerts (system C default configuration, risk threshold %.2f) by rule:" % cfg["alerts"]["risk_threshold"])

**Experiment 5 - ablations.** Remove one component at a time: dependency (graph) features, temporal features, evidence features, GenAI extraction, evidence verification. Feature-ablated models are retrained with the same family/hyper-parameters.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from src.evaluation import paired_bootstrap_diff, binary_metrics
te, yte = binary(lab.dense()["test"]); d = te.assign(_y=yte)
rows = []
for name in ("C_graph_ml", "C_no_graph_dependency", "C_no_temporal", "C_no_evidence", "C_no_graph_temporal", "B_independent_ml"):
    m = scorers[name]; p = m.predict(te); d[name] = p; mm = binary_metrics(yte, p)
    rows.append({"system": name, "n_features": len(m.features), "pr_auc": mm["pr_auc"], "roc_auc": mm["roc_auc"], "brier": mm["brier"], "ece": mm["ece"]})
tab = pd.DataFrame(rows)
tab["pr_auc_diff_vs_C"] = [0.0] + [paired_bootstrap_diff(d, "patient_id", lambda s, n=n: average_precision_score(s._y, s[n]), lambda s: average_precision_score(s._y, s["C_graph_ml"]), n_boot=lab.p["bootstrap"], seed=31)["diff"] for n in tab.system[1:]]
tab["diff_ci_lo"] = [0.0] + [paired_bootstrap_diff(d, "patient_id", lambda s, n=n: average_precision_score(s._y, s[n]), lambda s: average_precision_score(s._y, s["C_graph_ml"]), n_boot=lab.p["bootstrap"], seed=31)["lo"] for n in tab.system[1:]]
tab["diff_ci_hi"] = [0.0] + [paired_bootstrap_diff(d, "patient_id", lambda s, n=n: average_precision_score(s._y, s[n]), lambda s: average_precision_score(s._y, s["C_graph_ml"]), n_boot=lab.p["bootstrap"], seed=31)["hi"] for n in tab.system[1:]]
display(tab.round(4)); lab.save_csv("exp5_feature_ablation_dense_test", tab)
ewa = ew[(ew.min_lead_h == 24.0) & (ew["target_recall(val)"] == 0.7)][["system", "recall", "precision", "false_alert_rate", "median_lead_h"]]
print("Replay-level effect of each ablation (alerts >=24h before deadline, val-chosen threshold at recall 0.7):"); display(ewa[ewa.system.str.startswith(("C_", "B_"))].round(3))

In [ ]:
from src.exp_extraction import run_exp5_genai
t = time.time(); GEN = run_exp5_genai(lab, "primary"); print("GenAI end-to-end ablation took", round(time.time() - t), "s on", GEN["table"].episodes.iloc[0], "synthetic episodes")
display(GEN["table"].round(3))
print("'none' = no obligation extraction -> nothing is monitored; failures_surfaced counts true overdue obligations for which the monitoring system raised a reactive alert on the MATCHED extracted obligation. "
      "Spurious or mis-timed extractions raise false alerts. Outcomes in these episodes are simulated from explicit event timelines.")

In [ ]:
from src.exp_replay import run_priority
PRI = run_priority(lab, scorers); display(PRI["table"].round(3)); print("tuned weights:", {k: round(v, 3) for k, v in PRI["tuned_weights"].items()}, "| search budget:", PRI["search_budget"])
print("learned coefficients:", {k: round(v, 2) for k, v in PRI["learned_coefficients"].items()}); print("paired differences vs risk-only:", {k: (round(v["diff"], 3), round(v["lo"], 3), round(v["hi"], 3)) for k, v in PRI["paired"].items()}); print(PRI["note"])

In [ ]:
GENT = GEN["table"].set_index("extractor")
abl = pd.DataFrame([
    {"component removed": "graph dependency features", "evidence": f"PR-AUC {tab.set_index('system').loc['C_no_graph_dependency','pr_auc_diff_vs_C']:+.3f} (95% CI {tab.set_index('system').loc['C_no_graph_dependency','diff_ci_lo']:+.3f}..{tab.set_index('system').loc['C_no_graph_dependency','diff_ci_hi']:+.3f}) on dense test"},
    {"component removed": "temporal features", "evidence": f"PR-AUC {tab.set_index('system').loc['C_no_temporal','pr_auc_diff_vs_C']:+.3f} (95% CI {tab.set_index('system').loc['C_no_temporal','diff_ci_lo']:+.3f}..{tab.set_index('system').loc['C_no_temporal','diff_ci_hi']:+.3f})"},
    {"component removed": "evidence features", "evidence": f"PR-AUC {tab.set_index('system').loc['C_no_evidence','pr_auc_diff_vs_C']:+.3f} (95% CI {tab.set_index('system').loc['C_no_evidence','diff_ci_lo']:+.3f}..{tab.set_index('system').loc['C_no_evidence','diff_ci_hi']:+.3f})"},
    {"component removed": "evidence verification (engine -> naive absence=failure)", "evidence": f"state accuracy at t*: {se['engine_evidence_aware']['accuracy']:.3f} -> {se['naive_absence_is_failure']['accuracy']:.3f}; not-confirmed recall {se['engine_evidence_aware']['per_class_recall']['not_confirmed']:.2f} -> {se['naive_absence_is_failure']['per_class_recall']['not_confirmed']:.2f}"},
    {"component removed": "GenAI extraction (E -> rules only A)", "evidence": "failure recall %.3f -> %.3f; monitored fraction %.3f -> %.3f" % (GENT.loc["E_rules_plus_llm", "failure_recall"], GENT.loc["A_rules", "failure_recall"], GENT.loc["E_rules_plus_llm", "monitored_fraction"], GENT.loc["A_rules", "monitored_fraction"])},
])
display(abl); lab.save_csv("exp5_ablation_summary", abl)

## 15. Explainability and case-level error analysis

In [ ]:
from src.exp_replay import error_analysis
EA = error_analysis(lab, E3, k=3)
print(f"threshold {EA['threshold']:.3f} (validation-chosen, recall target 0.7, >=24h lead); cases scored at T-{EA['horizon_h']:.0f}h")
display(EA["cases"].drop(columns=["events (type, event time, ingest time)"]))
for _, r in EA["cases"].head(2).iterrows(): print(r["case"], r["obligation"], "->", r["events (type, event time, ingest time)"])
display(EA["by_profile"].round(3))
print("The hidden generator profile is used ONLY here, to see which simulated behaviours the model gets wrong; it is never a feature.")

In [ ]:
gi_c = gi.set_index("feature_group").pr_auc_drop_mean.rename("PR-AUC drop when group shuffled")
display(gi_c.to_frame().round(4)); print("Most informative group:", gi_c.idxmax())
oid = EA["cases"].iloc[0]["obligation"]
print("Audit trail for", oid); display(E3["processor_test"].audit_df(oid)[["t_ingest", "old_state", "new_state", "rule", "trigger_event_id", "evidence"]])
print([ (a.rule, a.status, a.explanation[:260]) for a in E3["processor_test"].alert_log if a.oid == oid][:2])

## 16. LinuxONE performance benchmarks

Methodology: warm-up discarded, repeated measurements, each throughput workload in a **fresh subprocess** (clean peak-RSS attribution). **ML-model latency and LLM generation latency are reported separately.**
Everything below is measured on the machine identified in section 2. "Event loss" is reported for a bounded queue under offered load.

In [ ]:
import gc, psutil
for _n in ("proc", "p2", "rec", "fin", "fm", "f", "pipe", "pipe2", "demo", "d", "gp", "g", "g1", "g2", "g3", "E4", "VM"):
    globals().pop(_n, None)
E3["processor_test"] = None; E3["scorers"] = None; gc.collect()
print("kernel RSS after releasing replay state: %.0f MB | system available: %.0f MB" % (psutil.Process().memory_info().rss / 1e6, psutil.virtual_memory().available / 1e6))

In [ ]:
from src import benchmarks as bm
from src.clinical_extractor import LLM_SCHEMA, SYSTEM_PROMPT
rows_f = te[mC.features].head(256).to_dict("records")
mlat = []
for nm in ("C_graph_ml", "B_independent_ml"):
    df_ = bm.bench_ml_latency(scorers[nm], rows_f, repeats=3 if MODE == "fast" else 5); df_.insert(0, "model", f"{nm} ({scorers[nm].kind})"); mlat.append(df_)
mlat = pd.concat(mlat); display(mlat.round(3))
conc = bm.bench_ml_concurrency(scorers["C_graph_ml"], rows_f, threads=(1, 2, 4, 8)); display(conc.round(2))

In [ ]:
prompts = [d["text"] for d in E1["corpora"]["developer_annotated_challenge"][:3 if MODE == "fast" else 6]]
llm_b = bm.bench_llm(be.backend, prompts, LLM_SCHEMA, SYSTEM_PROMPT, max_tokens=200)
display(pd.DataFrame({"value": {k: (round(v, 2) if isinstance(v, float) else v) for k, v in {"model": llm_b["model"], "model_size_mb": llm_b["model_size_mb"], "threads": llm_b["threads"], "load_s": llm_b["load_s"],
      "prompt_tokens_mean": llm_b["prompt_tokens_mean"], "completion_tokens_mean": llm_b["completion_tokens_mean"], "prompt_eval_s (1-token call)": llm_b["prompt_eval_s_first_token_proxy"],
      "generation latency p50 (s)": llm_b["latency_s"]["p50"], "p95 (s)": llm_b["latency_s"]["p95"], "p99 (s)": llm_b["latency_s"]["p99"], "tokens/s mean": llm_b["tokens_per_s"]["mean"], "peak RSS MB": llm_b["peak_rss_mb"]}.items()}}))
lconc = bm.bench_llm_concurrency(be.backend, prompts[0], LLM_SCHEMA, SYSTEM_PROMPT, concurrency=(1, 2, 4), max_tokens=40); display(lconc.round(2))
print("ML scoring latency (ms, section above) vs LLM generation latency (s): about", round(llm_b['latency_s']['p50'] * 1000 / mlat[(mlat.batch == 1)].latency_ms_p50.iloc[0]), "x slower per call - the LLM is never on the per-event path.")

In [ ]:
spec = cfg["benchmarks"][MODE]; reps = spec["repeats"]
bench_rows = []
bm.run_workload({"n_events": 500, "scoring": "ml", "storage": "memory", "rescore_batch": 64, "mode": MODE})        # warm-up (discarded)
configs = [("graph+states only (in-memory)", dict(scoring="none", storage="memory", rescore_batch=64, alerts=False)),
           ("states+alerts (in-memory)", dict(scoring="none", storage="memory", rescore_batch=64, alerts=True)),
           ("states+alerts+ML risk (in-memory, batch 64)", dict(scoring="ml", storage="memory", rescore_batch=64, alerts=True)),
           ("states+alerts+ML risk (SQLite WAL, commit/200, batch 64)", dict(scoring="ml", storage="sqlite", rescore_batch=64, alerts=True, commit_every=200)),
           ("states+alerts+ML risk (SQLite, per-event rescoring, commit/200)", dict(scoring="ml", storage="sqlite", rescore_batch=1, alerts=True, commit_every=200))]
for n in spec["event_workloads"]:
    for label, c in configs:
        if n > 10000 and (c["rescore_batch"] == 1 or label.startswith("states+alerts (")):
            continue                                   # the 100k workload only runs the three most informative configurations (wall-clock budget)
        for r in range(reps if n < 100000 else 1):     # 100k events: a single measured run per configuration
            res = bm.run_workload({"n_events": n, "mode": MODE, **c})
            bench_rows.append({"workload_events": n, "config": label, "run": r, **{k: res.get(k) for k in ("events", "wall_s", "events_per_s", "peak_rss_mb", "error_rate", "rejected", "alerts", "transitions")},
                               "lat_p50_ms": res.get("latency_ms", {}).get("p50"), "lat_p95_ms": res.get("latency_ms", {}).get("p95"), "lat_p99_ms": res.get("latency_ms", {}).get("p99"), "error": res.get("error")})
BR = pd.DataFrame(bench_rows); BR.to_csv("results/throughput_benchmark_runs.csv", index=False)
agg = BR.groupby(["workload_events", "config"]).agg(events_per_s=("events_per_s", "median"), eps_min=("events_per_s", "min"), eps_max=("events_per_s", "max"), lat_p50_ms=("lat_p50_ms", "median"),
                                                  lat_p95_ms=("lat_p95_ms", "median"), lat_p99_ms=("lat_p99_ms", "median"), peak_rss_mb=("peak_rss_mb", "max"), error_rate=("error_rate", "max"), runs=("run", "count")).reset_index()
display(agg.round(2))

In [ ]:
ev_q = events_test[:2000]
def make_proc():
    return EventProcessor(cfg, ":memory:", scorers={"C_graph_ml": scorers["C_graph_ml"]}, primary="C_graph_ml", rescore_batch=64, enable_alerts=True, log_risk=False, build_nx=False)
# capacity of THIS consumer (states + alerts + calibrated ML risk, micro-batch 64) measured on the same events, single thread, no queue
caps = []
for _ in range(3):                                       # first pass is a warm-up (model/lib first-call costs); capacity = best warm pass
    _p = make_proc(); _t = time.perf_counter()
    for _e in ev_q: _p.process(_e)
    _p.flush(); caps.append(len(ev_q) / (time.perf_counter() - _t)); del _p
cap = max(caps[1:])
qrows = []
for label, mult, drop, maxsize in (("0.5x capacity, block", 0.5, False, 200), ("2x capacity, block (backpressure)", 2.0, False, 200), ("2x capacity, drop (bounded queue 200)", 2.0, True, 200), ("4x capacity, drop (bounded queue 200)", 4.0, True, 200)):
    p = make_proc(); r = bm.bench_queue(p.process, ev_q, rate_eps=cap * mult, maxsize=maxsize, drop_when_full=drop, finalize_fn=p.flush); r["scenario"] = label; qrows.append(r)
display(pd.DataFrame(qrows)[["scenario", "offered_rate_eps", "produced", "consumed", "dropped", "loss_rate", "max_backlog", "queue_latency_ms_p50", "queue_latency_ms_p95", "queue_latency_ms_p99", "produce_wall_s", "ideal_produce_wall_s", "total_wall_s", "policy"]].round(3))
print(f"consumer capacity measured on the same events (best of 2 warm passes {[round(c) for c in caps]}): {cap:.0f} events/s (states+alerts+ML risk). Offered rate = multiple of it. 'block' = backpressure (producer stalls, nothing lost: compare produce_wall_s with ideal_produce_wall_s); 'drop' = bounded queue sheds events.")

In [ ]:
mm = bm.nnpa_matmul_bench()
if mm["available"]:
    nd = pd.DataFrame(mm["rows"])[["M", "K", "N", "nnpa_op_ms", "nnpa_total_ms", "numpy_blas_ms", "cpu_naive_c_ms", "nnpa_op_gflops", "numpy_blas_gflops", "speedup_op_vs_numpy", "speedup_total_vs_numpy", "max_abs_err", "max_abs_ref"]]
    display(nd.round(4)); print("zDNN", mm["zdnn_version"], "-", mm["interpretation"])
else:
    print("NNPA device test not run on this host:", mm["reason"])

In [ ]:
import platform as _pl
bench_out = {"host": {k: slim[k] for k in ("uname_m", "byteorder", "nproc", "memory_total_gb", "hypervisor", "python")}, "ml_latency": mlat.to_dict("records"), "ml_concurrency": conc.to_dict("records"), "llm": llm_b,
             "llm_concurrency": lconc.to_dict("records"), "throughput": agg.to_dict("records"), "queue": qrows, "nnpa_matmul": mm, "nnpa_probe": nn, "mode": MODE}
Path("results/benchmarks.json").write_text(json.dumps(bench_out, indent=2, default=str))
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
a2 = agg.pivot(index="config", columns="workload_events", values="events_per_s"); a2.plot(kind="barh", ax=ax[0]); ax[0].set_title("Event-processing throughput (events/s, median)"); ax[0].legend(title="events", fontsize=7)
a3 = agg.pivot(index="config", columns="workload_events", values="peak_rss_mb"); a3.plot(kind="barh", ax=ax[1]); ax[1].set_title("Peak resident memory (MB)"); ax[1].legend(title="events", fontsize=7)
mlat.pivot(index="batch", columns="model", values="latency_ms_p50").plot(ax=ax[2], marker="o"); ax[2].set_xscale("log"); ax[2].set_title("ML scoring latency p50 (ms) vs batch size"); ax[2].set_ylabel("ms")
plt.tight_layout(); viz.save(fig, cfg, "benchmarks"); plt.show()

## 17. Interactive live demonstration

**Required scenario (synthetic patient):** discharged with (1) a laboratory test within 48 h, (2) physician result review, (3) conditional patient notification, (4) a follow-up appointment.
The test completes, but the acknowledgment evidence is **delayed**. The system recognises the dependency chain, separates *completed* from *unverified*, updates the forecast risk as events stream in, raises an evidence-grounded alert,
shows the graph and timeline, **reconciles the state when the delayed acknowledgment is finally documented**, and keeps the full audit trail. The same engine, models and rules as the experiments are used (streaming inference at every event).

In [ ]:
from src.visualization_demo import LiveDemo, audit_table, alert_feed_html, plot_patient_journey, plot_evidence_graph, plot_risk_monitor
from src.scenario_lab import demo_events, DEMO_TEXT
live_scorers = {"B_independent_ml": scorers["B_independent_ml"], "C_graph_ml": scorers["C_graph_ml"]}
evs = demo_events("delayed_ack"); demo = LiveDemo(cfg, live_scorers, evs, PID)
idx = {e["event_id"]: i + 1 for i, e in enumerate(evs)}
print("scripted deliveries (ingest order):"); display(pd.DataFrame([{"#": i + 1, "event_type": e["event_type"], "obligation": e["obligation_id"].split(":")[-1], "event_time": e["event_time_utc"], "ingest_time": e["ingest_time_utc"]} for i, e in enumerate(evs)]))

In [ ]:
steps = sorted({7, 9, 10, len(evs)})
for k, p, fj, fg in demo.storyboard(steps, with_graph=True):
    print(f"=== frame after {k}/{len(evs)} deliveries (service clock {iso(p.clock)}) ===")
    display(fj); display(fg); plt.close(fj); plt.close(fg)
    display(pd.DataFrame([{"obligation": o.split(":")[-1], "state": p.graph.derive_state(o, p.clock, live=True).state.value, "risk(C)": round((p.last_risk.get(o) or {}).get("C_graph_ml", float("nan")), 2)} for o in sorted(p.graph.obs)]))
    display(HTML(alert_feed_html(p, PID, last=4)))

In [ ]:
fin = demo.run_to(len(evs)); fin.finalize()
fig = plot_risk_monitor(fin, PID, "C_graph_ml"); viz.save(fig, cfg, "demo_risk_monitor"); plt.show()
print("AUDIT TRAIL (every state change with event time/ingest time, rule and evidence):"); display(audit_table(fin, PID))
print("final alert feed (newest first):"); display(HTML(alert_feed_html(fin, PID, last=10)))
ob1 = [t for t in fin.transitions if t["oid"] == "demo:ob1"]
assert [t["new_state"] for t in ob1][-2:] == ["overdue_without_verified_completion", "completed_with_evidence"], "delayed acknowledgment was not reconciled"
print("RECONCILIATION VERIFIED: review obligation went overdue_without_verified_completion -> completed_with_evidence when the delayed acknowledgment arrived; late flag =", fin.states["demo:ob1"][2])

In [ ]:
# variants: acknowledgment NEVER documented, and the happy path
for v in ("missing_ack", "happy_path"):
    d = LiveDemo(cfg, live_scorers, demo_events(v), PID); f = d.run_to(len(d.events)); f.finalize()
    print("variant:", v, "->", {o.split(":")[-1]: f.states[o][0].value + (" (late)" if f.states[o][2] else "") for o in sorted(f.states)})
fig = plot_patient_journey(f, PID, "happy path"); plt.show()
fm = LiveDemo(cfg, live_scorers, demo_events("missing_ack"), PID).run_to(99); fm.finalize()
fig = plot_patient_journey(fm, PID, "Missing acknowledgment: downstream tasks stay unverified/overdue - the completed test does not complete the chain"); plt.show()
display(HTML(alert_feed_html(fm, PID, last=6)))

In [ ]:
# interactive widget (falls back to the static storyboard above when widgets cannot render, e.g. in a headless run)
try:
    import ipywidgets
    from IPython import get_ipython
    if get_ipython() is not None and os.environ.get("MEDRELAY_HEADLESS", "0") != "1":
        LiveDemo(cfg, live_scorers, demo_events("delayed_ack"), PID).widget()
    else:
        print("headless execution: interactive widget skipped; the storyboard frames above show the same states.")
except Exception as e:
    print("ipywidgets unavailable (", type(e).__name__, ") - using the static storyboard above")

In [ ]:
display(HTML("<b>Performance monitor (measured on %s, mode=%s)</b>" % (hw["uname_m"], MODE)))
best_row = agg.sort_values("events_per_s", ascending=False).iloc[0]
display(pd.DataFrame({"metric": ["ML scoring latency p50 / p95 / p99, batch 1 (ms)", "ML scoring throughput, batch 256 (rows/s)", "event processing throughput, best config (events/s)", "LLM generation p50 / p95 (s)", "LLM tokens/s", "peak RSS, event workload (MB)", "queue event loss, block policy"],
                      "value": [" / ".join(f"{x:.1f}" for x in mlat[(mlat.model.str.startswith("C_")) & (mlat.batch == 1)][["latency_ms_p50", "latency_ms_p95", "latency_ms_p99"]].iloc[0]), f"{mlat[(mlat.model.str.startswith('C_')) & (mlat.batch == 256)].rows_per_s.iloc[0]:.0f}",
                                f"{best_row.events_per_s:.0f} ({best_row.config}, {best_row.workload_events} events)", f"{llm_b['latency_s']['p50']:.1f} / {llm_b['latency_s']['p95']:.1f}", f"{llm_b['tokens_per_s']['mean']:.1f}",
                                f"{agg.peak_rss_mb.max():.0f}", str(qrows[0]['event_loss']) + " / " + str(qrows[1]['event_loss'])]}))

## 18. Results, limitations and final conclusions

In [ ]:
import json as _j
def load(n): return _j.loads(Path(f"results/{n}.json").read_text())
mA = A_res["comparison"].sort_values("val_macro_f1", ascending=False).iloc[0]
cB = B_res["comparison"]; bC = cB[cB.features == "C_graph"].sort_values("val_pr_auc", ascending=False).iloc[0]; bB = cB[cB.features == "B_independent"].sort_values("val_pr_auc", ascending=False).iloc[0]
mC_ = load("model_c_summary"); cC = pd.DataFrame(mC_["comparison"]).sort_values("val_pr_auc", ascending=False).iloc[0]
r1 = E1["table"]; ew24 = ew[(ew.min_lead_h == 24.0) & (ew["target_recall(val)"] == 0.7)].set_index("system")
summary = pd.DataFrame([
    ["Action classification (silver labels)", f"{mA.model} macro-F1 {mA.test_macro_f1:.3f} / micro-F1 {mA.test_micro_f1:.3f} (test); developer-annotated spans: accuracy {mA.challenge_span_accuracy:.3f}"],
    ["Follow-up risk (simulated outcomes)", f"best graph model {bC.model}: PR-AUC {bC.test_pr_auc:.3f} [{bC.test_pr_auc_ci_lo:.3f},{bC.test_pr_auc_ci_hi:.3f}], ROC-AUC {bC.test_roc_auc:.3f}, Brier {bC.test_brier:.3f}, ECE {bC.test_ece:.3f}; independent-task {bB.model}: PR-AUC {bB.test_pr_auc:.3f}"],
    ["No-show (real data, secondary)", f"{cC.model}: PR-AUC {cC.test_pr_auc:.3f} (prevalence {cC.prevalence:.3f}), ROC-AUC {cC.test_roc_auc:.3f}, ECE {cC.test_ece:.3f}"],
    ["GenAI extraction (350M)", "typed F1 B/C2/E on challenge set: " + " / ".join(f"{r1[(r1.extractor == x) & (r1.corpus == 'developer_annotated_challenge')].typed_f1.iloc[0]:.2f}" for x in ('B_llm_raw', 'C2_llm_validated_filled', 'E_rules_plus_llm')) + f"; rules A {r1[(r1.extractor == 'A_rules') & (r1.corpus == 'developer_annotated_challenge')].typed_f1.iloc[0]:.2f}"],
    ["Early warning (>=24h lead, recall target 0.7 on val)", " | ".join(f"{s}: recall {ew24.loc[s,'recall']:.2f}, FAR {ew24.loc[s,'false_alert_rate']:.2f}, lead {ew24.loc[s,'median_lead_h']:.0f}h" for s in ("A_reminder_24h", "A_deadline_only", "B_independent_ml", "C_graph_ml"))],
    ["State correctness at t*", f"engine {se['engine_evidence_aware']['accuracy']:.3f} (final after reconciliation {se['engine_final_after_reconciliation']['accuracy']:.3f}) vs naive absence=failure {se['naive_absence_is_failure']['accuracy']:.3f}"],
    ["Throughput (best config)", f"{best_row.events_per_s:.0f} events/s on {hw['uname_m']} ({hw['nproc']} vCPU)"]], columns=["result", "measured value"])
display(summary); summary.to_csv("results/final_summary.csv", index=False)
print("notebook wall time: %.1f min | mode=%s | host=%s" % ((time.time() - T_NB0) / 60, MODE, hw["uname_m"]))

### Interpretation, limitations and honest conclusions

* **Research question (simulated processes only).** The tables above quantify, with patient-grouped bootstrap intervals, whether dependency-aware graph features improve *early* warning over independent-task monitoring. Read the lead-restricted rows
  (>= 24 h / 72 h before the deadline): a deadline-only rule is hard to beat *late* in the window (it only fires once most of the window has elapsed), so any advantage of the graph approach is an *early-warning* advantage; see the confidence intervals before
  concluding anything, and note when an interval includes zero.
* **Labels.** Risk and state outcomes come from a **seeded simulator** whose causal structure we know; scores say how well the pipeline recovers that structure, **not** how it would perform on real hospital data. Hidden generator profiles are used only for post-hoc error analysis.
* **No gold annotations.** Action labels are rule-proposed; extraction evaluation uses a *developer-annotated* challenge set and synthetic-by-construction documents. Independent clinician review is the single most valuable missing piece.
* **GenAI.** The 350M Granite model is too weak to extract obligations on its own; validation removes fabricated fields but cannot add missing obligations. The ensemble with rules is the best measured GenAI-involving strategy; the 1B model (smaller sample) is the measured upper bound. Latency is dominated by the LLM, so it is kept off the per-event path.
* **Hardware.** Section 2 states the architecture that executed this run; throughput/latency are for that VM (2 vCPU, ~4 GB). NNPA is *present* and was device-tested with zDNN matmul; the tree/linear models do **not** use it.
* **Not a medical device.** All recommendations are for qualified human review; nothing here modifies orders, prescriptions, diagnoses or treatment.